# 01 — First look at the London crime data

Goal of this notebook: load the raw CSVs and understand what we're actually
working with — shape, columns, dtypes, missing values, date coverage — before
we do any real analysis. No conclusions yet, just honest observation.

In [ ]:
# sys.path.append lets us import from src/, two levels up from
# notebooks/london/ (notebooks/london/ -> notebooks/ -> project root -> src/).
import sys
sys.path.append("../../src")

import pandas as pd
from load_data import load_force_data

london = load_force_data("london")
london.shape

`.shape` gives (rows, columns) — a quick sanity check that we got roughly
what we expected (13 months of ~90-100k rows each).

In [2]:
london.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context,source_file
0,ad9425ef998a2e0de31752452ec20311ee771d9ffd8b93...,2025-05,Metropolitan Police Service,Metropolitan Police Service,-0.612732,50.816388,On or near Blenheim Road,E01031473,Arun 006D,Violence and sexual offences,Status update unavailable,NaN,2025-05-metropolitan-street.csv
1,NaN,2025-05,Metropolitan Police Service,Metropolitan Police Service,0.134947,51.588063,On or near Mead Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,2025-05-metropolitan-street.csv
2,NaN,2025-05,Metropolitan Police Service,Metropolitan Police Service,0.136416,51.584898,On or near Lawn Farm Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,2025-05-metropolitan-street.csv
3,NaN,2025-05,Metropolitan Police Service,Metropolitan Police Service,0.142112,51.589389,On or near A1112,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,2025-05-metropolitan-street.csv
4,NaN,2025-05,Metropolitan Police Service,Metropolitan Police Service,0.140127,51.588913,On or near Beansland Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN,2025-05-metropolitan-street.csv


In [3]:
# .info() shows column names, dtypes, and non-null counts in one go —
# usually the first thing worth running on any new DataFrame.
london.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1235489 entries, 0 to 1235488
Data columns (total 13 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Crime ID               976435 non-null   object 
 1   Month                  1235489 non-null  object 
 2   Reported by            1235489 non-null  object 
 3   Falls within           1235489 non-null  object 
 4   Longitude              1235489 non-null  float64
 5   Latitude               1235489 non-null  float64
 6   Location               1235489 non-null  object 
 7   LSOA code              1235489 non-null  object 
 8   LSOA name              1235489 non-null  object 
 9   Crime type             1235489 non-null  object 
 10  Last outcome category  976435 non-null   object 
 11  Context                0 non-null        float64
 12  source_file            1235489 non-null  object 
dtypes: float64(3), object(10)
memory usage: 122.5+ MB


In [4]:
# .isna() marks each cell True/False for missing; .sum() adds those up
# per column (True counts as 1). This tells us which columns actually
# have gaps, and how big they are.
london.isna().sum()

Crime ID                  259054
Month                          0
Reported by                    0
Falls within                   0
Longitude                      0
Latitude                       0
Location                       0
LSOA code                      0
LSOA name                      0
Crime type                     0
Last outcome category     259054
Context                  1235489
source_file                    0
dtype: int64

### Findings (confirmed, not assumed)
- `Crime ID` is missing for ~21% of rows, and **every single one is
  "Anti-social behaviour"** — confirmed by grouping Crime type where
  Crime ID is null. This is deliberate anonymisation by the police, not
  a loading bug. `Last outcome category` is null for the exact same rows,
  since ASB incidents don't get formal tracked outcomes.
- ~6,899 Crime IDs appear more than once. Checked an example: same ID,
  same month, but two different Crime types ("Other theft" and
  "Criminal damage and arson"). This is one real incident filed under
  multiple offence categories — not double-counted data. Decide
  deliberately later whether to keep both rows (every offence counted)
  or collapse to one row per incident, depending on the question.
- `Context` is 100% empty across all 1.2M rows — safe to drop.

In [5]:
# .unique() lists distinct values. For Month, this confirms exactly which
# months made it into the combined DataFrame, in what looks like order
# (though unique() doesn't guarantee sorted output).
sorted(london["Month"].unique())

['2025-05',
 '2025-06',
 '2025-07',
 '2025-08',
 '2025-09',
 '2025-10',
 '2025-11',
 '2025-12',
 '2026-01',
 '2026-02',
 '2026-03',
 '2026-04',
 '2026-05']

In [6]:
# Are any Crime IDs duplicated? Real crimes shouldn't appear twice.
# We exclude blank IDs first since many rows share the same "missing" value,
# which would otherwise look like massive duplication.
has_id = london["Crime ID"].dropna()
has_id.duplicated().sum()

np.int64(6899)

In [7]:
# value_counts() tallies how often each distinct value appears —
# here, which crime types are most common in London.
london["Crime type"].value_counts()

Crime type
Violence and sexual offences    301957
Anti-social behaviour           259054
Other theft                     106556
Shoplifting                      97233
Vehicle crime                    89217
Theft from the person            83393
Public order                     65615
Drugs                            56727
Criminal damage and arson        54516
Burglary                         50981
Robbery                          33695
Bicycle theft                    15107
Other crime                      14314
Possession of weapons             7124
Name: count, dtype: int64

## Cleaning: drop columns we don't need

- `Reported by` / `Falls within` — constant ("Metropolitan Police Service")
  for every row in this dataset, so they carry no information.
- `LSOA code` — keeping `LSOA name` instead, since it's human-readable.
- `Context` — confirmed 100% empty above.
- `source_file` — only added by our loader for debugging; not needed now.

In [ ]:
# drop(columns=[...]) returns a NEW DataFrame with those columns removed —
# it does NOT modify `london` in place unless we pass inplace=True (which
# most style guides now discourage, since reassignment is more explicit
# about the fact that something changed).
london = london.drop(columns=[
    "Reported by",
    "Falls within",
    "LSOA code",
    "Context",
    "source_file",
])

london.columns

## Viewing both rows of a repeated Crime ID

Earlier we counted 6,899 repeated Crime IDs using `.duplicated()`, which by
default only flags the **second-and-later** occurrence as `True` — the first
occurrence of each duplicated value stays hidden as `False`. That's fine for
counting, but if we want to actually *see* every row involved in a repeat, we
need `keep=False`, which flags **all** copies (first included) as `True`.

In [ ]:
# notna() first, so blank Crime IDs (the ASB rows) don't get treated as
# one giant "duplicate" group of their own.
# keep=False marks EVERY row sharing a repeated value as True (not just
# the later ones) -> this is what "brings back" the first occurrence too.
has_id = london["Crime ID"].notna()
is_repeated = london["Crime ID"].duplicated(keep=False)

repeated_crimes = london[has_id & is_repeated]

# sort_values groups each pair/group of matching Crime IDs together in the
# output, so you can visually compare the rows side by side.
repeated_crimes = repeated_crimes.sort_values("Crime ID")

repeated_crimes.head(10)